# Tutorial 9: Backtesting — Synthetic & Historical

Training tells you if the agent is learning. Backtesting tells you if it would actually make money.

Spectral-Env-Core provides two backtesting modes:

1. **Synthetic** — run the trained model on 100+ stochastic episodes. Tests robustness across the full distribution of market characters (different regimes, GARCH realisations, jump events).

2. **Historical** — fetch real price data for the same tickers and replay it through the environment. Tests whether the agent's learned strategy transfers to actual market history it was never trained on.

Both modes use identical environment mechanics (fees, slippage, terminal liquidation) — only the price source changes.

---

In [ ]:
import numpy as np
from stable_baselines3 import PPO
from spectral_env_core import (
    SpectralTradingEnv,
    SpectralExtractor,
    EntropyCoefficientCallback,
    estimate_params,
    backtest,
    backtest_historical,
)

## Step 1: Train an Agent

First we need a trained model to backtest. Let's calibrate to NVDA and train a quick agent.

In [ ]:
# Calibrate environment to NVDA
params = estimate_params("NVDA", period="3y")

env_kwargs = {
    **params,
    "num_steps": 252,
    "starting_cash": 50_000,
    "max_shares": 200,
    "max_trade_size": 20,
    "transaction_cost_pct": 0.0005,
    "slippage_bps": 5.0,
    "liquidity_shares": 5_000,
}

env = SpectralTradingEnv(**env_kwargs)
print(f"Training env calibrated to NVDA @ ${params['initial_price']}")
print(f"With slippage: 5 bps, liquidity depth: 5000 shares")

In [ ]:
# Train (short run for demo — use 3-7M steps for real training)
ent_cb = EntropyCoefficientCallback(start=0.005, end=0.001)

model = PPO(
    "MlpPolicy", env, verbose=0,
    learning_rate=3e-4, n_steps=2048, batch_size=64,
    n_epochs=10, gamma=0.99, ent_coef=0.005,
)

print("Training for 300K steps...")
model.learn(total_timesteps=300_000, callback=[ent_cb])
print("Done.")

---

## Step 2: Synthetic Backtest

Run the model on 100 stochastically generated episodes. Each episode has a unique regime sequence, GARCH path, and AR(1) character — testing whether the agent generalises or memorised a single market pattern.

In [ ]:
# One function call — runs 100 episodes, computes all stats
synthetic_report = backtest(
    model=model,
    env_kwargs=env_kwargs,
    n_episodes=100,
    deterministic=True,
    seed_start=5000,  # seeds never seen during training
)

print(synthetic_report.summary)

In [ ]:
# Visual report: equity curves, return distribution, worst drawdown, friction
synthetic_report.plot()

### Reading the Synthetic Report

| Metric | What it tells you |
|---|---|
| **Sharpe > 0** | Agent generates positive risk-adjusted returns |
| **Win rate > 50%** | More profitable episodes than losing ones |
| **Max drawdown** | Worst-case intra-episode loss — are you comfortable with this? |
| **Profit factor > 1** | Total gains exceed total losses |
| **Avg friction** | How much the agent pays in fees + slippage per episode |
| **Std return** | How consistent is the agent across different market realisations? |

A good synthetic backtest means the agent isn't overfitting to a single stochastic pattern — it performs across the full distribution of simulated markets.

---

## Step 3: Historical Backtest

The real test: does the agent perform on actual market data it was never trained on?

`backtest_historical` fetches real price data from yfinance and replays it through the exact same environment mechanics. The agent sees real NVDA price movements — with all the earnings surprises, macro events, and sentiment shifts that the stochastic model can only approximate.

In [ ]:
# Historical backtest on real NVDA data (last 2 years)
historical_report = backtest_historical(
    model=model,
    tickers=["NVDA"],
    period="2y",
    env_kwargs={
        "starting_cash": 50_000,
        "max_shares": 200,
        "max_trade_size": 20,
        "transaction_cost_pct": 0.0005,
        "slippage_bps": 5.0,
        "liquidity_shares": 5_000,
    },
    deterministic=True,
)

print(historical_report.summary)

In [ ]:
historical_report.plot()

### Comparing Synthetic vs Historical

The key question: **does performance transfer?**

In [ ]:
print(f"{'Metric':<20s}  {'Synthetic':>12s}  {'Historical':>12s}")
print('─' * 48)
for key in ['mean_return', 'sharpe_ratio', 'max_drawdown', 'win_rate', 'avg_friction']:
    syn = synthetic_report.stats.get(key, 0)
    hist = historical_report.stats.get(key, 0)
    if 'rate' in key:
        print(f"{key:<20s}  {syn:>11.1%}  {hist:>11.1%}")
    elif 'friction' in key:
        print(f"{key:<20s}  ${syn:>10,.2f}  ${hist:>10,.2f}")
    else:
        print(f"{key:<20s}  {syn:>12.3f}  {hist:>12.3f}")

### Interpreting the Gap

If synthetic >> historical:
- The stochastic model may not capture some real-world dynamics (news events, earnings, sector rotation)
- Consider adding more jump intensity or regime switches to training

If synthetic ≈ historical:
- Strong evidence that the calibrated stochastic model produces realistic training data
- The agent's learned patterns transfer to real markets

If historical >> synthetic:
- Unusual — the real market period may have been unusually favourable (strong bull run)
- The synthetic test is more conservative because it includes bear regimes and crashes

---

## Step 4: Multi-Asset Historical Backtest

If you trained a multi-asset agent, backtest it on the real correlated portfolio:

In [ ]:
# Example: 3-asset portfolio backtest (requires a multi-asset trained model)
# Uncomment and run if you have a multi-asset model:

# multi_report = backtest_historical(
#     model=multi_asset_model,
#     tickers=["AAPL", "MSFT", "NVDA"],
#     period="2y",
#     env_kwargs={
#         "starting_cash": 100_000,
#         "max_shares": 500,
#         "max_trade_size": 50,
#         "slippage_bps": 5.0,
#         "liquidity_shares": 10_000,
#     },
# )
# print(multi_report.summary)
# multi_report.plot()

print("Multi-asset historical backtest uses real correlated price data.")
print("Each 252-day window becomes one episode — the agent manages")
print("a real portfolio through actual market conditions.")

---

## How Historical Backtesting Works Internally

```
1. Fetch real prices: yfinance → OHLCV → close prices
2. Align all tickers to same date range
3. Slice into non-overlapping 252-day windows (each = 1 episode)
4. For each window:
   a. Create env with reset() to initialise all state
   b. Replace env.brownian_path with real price window
   c. Run model.predict() for 252 steps
   d. All mechanics (fees, slippage, terminal liquidation) operate normally
5. Compute stats across all episodes
```

The environment doesn't know or care that the prices are real — it just executes trades, applies fees and slippage, computes rewards. The only difference from training is the price source.

This means your slippage model, terminal liquidation, and reward signal are all tested on real data — not just the price prediction.

---

## The Full Workflow

```
1. estimate_params("NVDA") → calibrate stochastic model
2. Train on infinite synthetic episodes (robust generalisation)
3. backtest(model, env_kwargs, n_episodes=100) → synthetic validation
4. backtest_historical(model, tickers=["NVDA"], period="2y") → real-world validation
5. If both look good → deploy to paper trading
```

Step 3 tells you the agent isn't broken. Step 4 tells you it works in reality.